# Task 4 - Raw vs Platform Training Dataset Validation

This notebook validates that the raw-data workflow and the integrated-platform workflow
produce the same baseline training records.

The platform table includes the January 2025 incremental update, while the raw workflow
uses only the baseline January-March 2024 input files. Therefore, we expect:

- identical output schemas
- identical row counts for all baseline months
- additional January 2025 rows only in the platform output
- `trip_id` and `dataset_split` are not compared directly, because the raw and
  platform workflows assign IDs at different stages.

In [1]:
from pyspark.sql import SparkSession, functions as F

RAW_TRAINING_PATH = (
    "s3a://delta-data/predictions/raw_ml_tip_training"
)

PLATFORM_TRAINING_PATH = (
    "s3a://delta-data/predictions/gold_ml_tip_training"
)

spark = (
    SparkSession.builder
    .appName("task4-raw-vs-platform-validation")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension",
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .getOrCreate()
)

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sparkContext.setLogLevel("ERROR")

raw_training = (
    spark.read
    .format("delta")
    .load(RAW_TRAINING_PATH)
)

platform_training = (
    spark.read
    .format("delta")
    .load(PLATFORM_TRAINING_PATH)
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/03 14:37:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [2]:
raw_schema = {
    field.name: field.dataType.simpleString()
    for field in raw_training.schema.fields
}

platform_schema = {
    field.name: field.dataType.simpleString()
    for field in platform_training.schema.fields
}

print("Schemas are identical:", raw_schema == platform_schema)

schema_differences = [
    (
        column_name,
        raw_schema.get(column_name),
        platform_schema.get(column_name),
    )
    for column_name in sorted(set(raw_schema) | set(platform_schema))
    if raw_schema.get(column_name) != platform_schema.get(column_name)
]

if schema_differences:
    spark.createDataFrame(
        schema_differences,
        ["column", "raw_type", "platform_type"],
    ).show(truncate=False)
else:
    print("No schema differences found.")

raw_row_count = raw_training.count()
platform_row_count = platform_training.count()

print(f"Raw baseline workflow rows: {raw_row_count:,}")
print(f"Platform workflow rows: {platform_row_count:,}")
print("Additional platform rows:", f"{platform_row_count - raw_row_count:,}")

Schemas are identical: True
No schema differences found.


Raw baseline workflow rows: 9,360,061
Platform workflow rows: 9,996,643
Additional platform rows: 636,582


In [3]:
raw_monthly_counts = (
    raw_training
    .groupBy(
        F.date_format(
            "pickup_datetime",
            "yyyy-MM",
        ).alias("month")
    )
    .count()
    .withColumnRenamed("count", "raw_rows")
)

platform_monthly_counts = (
    platform_training
    .groupBy(
        F.date_format(
            "pickup_datetime",
            "yyyy-MM",
        ).alias("month")
    )
    .count()
    .withColumnRenamed("count", "platform_rows")
)

monthly_comparison = (
    raw_monthly_counts
    .join(platform_monthly_counts, "month", "full")
    .fillna(0, subset=["raw_rows", "platform_rows"])
    .withColumn(
        "platform_minus_raw",
        F.col("platform_rows") - F.col("raw_rows"),
    )
    .orderBy("month")
)

monthly_comparison.show(truncate=False)

+-------+--------+-------------+------------------+
|month  |raw_rows|platform_rows|platform_minus_raw|
+-------+--------+-------------+------------------+
|2003-01|3       |3            |0                 |
|2009-01|2       |2            |0                 |
|2024-01|2880471 |2880471      |0                 |
|2024-02|2937461 |2937461      |0                 |
|2024-03|3525849 |3525849      |0                 |
|2024-04|16275   |16275        |0                 |
|2025-01|0       |636582       |636582            |
+-------+--------+-------------+------------------+



In [4]:
BASELINE_END = "2025-01-01"

AIR_QUALITY_COLUMNS = [
    "average_value_pu",
    "average_value_do",
]

def compare_hourly_air_quality(
    timestamp_column: str,
    air_quality_column: str,
    label: str,
) -> None:
    raw_hourly = (
        raw_training
        .select(
            F.date_trunc(
                "hour",
                F.col(timestamp_column),
            ).alias("hour"),
            F.col(air_quality_column).alias("raw_value"),
        )
        .dropDuplicates(["hour"])
    )

    platform_hourly = (
        platform_training
        .filter(F.col("pickup_datetime") < F.lit(BASELINE_END))
        .select(
            F.date_trunc(
                "hour",
                F.col(timestamp_column),
            ).alias("hour"),
            F.col(air_quality_column).alias("platform_value"),
        )
        .dropDuplicates(["hour"])
    )

    comparison = (
        raw_hourly
        .join(platform_hourly, "hour", "inner")
        .withColumn(
            "absolute_difference",
            F.abs(F.col("raw_value") - F.col("platform_value")),
        )
    )

    print(label)

    comparison.agg(
        F.count("*").alias("matched_hours"),
        F.max("absolute_difference").alias(
            "maximum_absolute_difference",
        ),
        F.avg("absolute_difference").alias(
            "average_absolute_difference",
        ),
        F.sum(
            F.when(
                F.col("absolute_difference") > F.lit(1e-9),
                F.lit(1),
            ).otherwise(F.lit(0))
        ).alias("hours_different_above_1e_minus_9"),
    ).show(truncate=False)


compare_hourly_air_quality(
    "pickup_datetime",
    "average_value_pu",
    "Pickup Air Quality",
)

compare_hourly_air_quality(
    "dropoff_datetime",
    "average_value_do",
    "Dropoff Air Quality",
)

Pickup Air Quality


+-------------+---------------------------+---------------------------+--------------------------------+
|matched_hours|maximum_absolute_difference|average_absolute_difference|hours_different_above_1e_minus_9|
+-------------+---------------------------+---------------------------+--------------------------------+
|2189         |7.105427357601002E-15      |4.158252903586896E-16      |0                               |
+-------------+---------------------------+---------------------------+--------------------------------+

Dropoff Air Quality


+-------------+---------------------------+---------------------------+--------------------------------+
|matched_hours|maximum_absolute_difference|average_absolute_difference|hours_different_above_1e_minus_9|
+-------------+---------------------------+---------------------------+--------------------------------+
|2210         |7.105427357601002E-15      |4.1688773828483645E-16     |0                               |
+-------------+---------------------------+---------------------------+--------------------------------+



In [5]:
comparison_columns = [
    column_name
    for column_name in raw_training.columns
    if column_name not in {"dataset_split", "trip_id"}
]

NORMALIZED_COMPARISON_COLUMNS = [
    (
        F.round(F.col(column_name), 9).alias(column_name)
        if column_name in AIR_QUALITY_COLUMNS
        else F.col(column_name)
    )
    for column_name in comparison_columns
]

normalized_raw_records = raw_training.select(
    *NORMALIZED_COMPARISON_COLUMNS
)

normalized_platform_records = (
    platform_training
    .filter(F.col("pickup_datetime") < F.lit(BASELINE_END))
    .select(*NORMALIZED_COMPARISON_COLUMNS)
)

raw_only_after_rounding = (
    normalized_raw_records
    .exceptAll(normalized_platform_records)
    .count()
)

platform_only_after_rounding = (
    normalized_platform_records
    .exceptAll(normalized_raw_records)
    .count()
)

print("Rows only in raw after Air Quality rounding:", raw_only_after_rounding)
print("Rows only in platform after Air Quality rounding:", platform_only_after_rounding)

assert raw_only_after_rounding == 0
assert platform_only_after_rounding == 0

print(
    "Success: baseline records are equivalent after normalizing insignificant floating-point variation."
)

Rows only in raw after Air Quality rounding: 0
Rows only in platform after Air Quality rounding: 0
Success: baseline records are equivalent after normalizing insignificant floating-point variation.


In [6]:
spark.stop()

## Conclusion

The raw workflow and platform workflow produced identical schemas. The platform contains
636,582 additional January 2025 records from the incremental update, while the raw
workflow deliberately processes the baseline source files only.

For the shared baseline period, Taxi, Taxi Zone, and Weather features matched exactly.
Air Quality averages differed only by floating-point precision below 10^-9. After
rounding these values to nine decimal places, all baseline training records matched.